# AI101 Assignment 4 - Task 1 Financial Data

**Purpose:** Use Gemini to analyze synthetic company financial data and predict stock prices.

**Dataset note:** This project uses a small synthetic dataset created for practice. It is not real financial or vehicle-market data. The AI-generated predictions are demonstrations, not professional advice.

## 1. Install and import libraries
The first cell installs Google's current Gemini Python SDK (`google-genai`). The remaining imports handle tables, files, and graphs.

In [ ]:
!pip -q install google-genai pandas matplotlib

In [ ]:
import os
import re
import io
import getpass
import pandas as pd
import matplotlib.pyplot as plt
from google import genai

# In Colab, add GEMINI_API_KEY under the Secrets panel to avoid retyping it.
# If no Colab secret or environment variable is set, prompt privately.
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
if not api_key:
    api_key = getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"

# Quick connection check requested by the assignment handout.
demo_response = client.models.generate_content(
    model=MODEL_NAME,
    contents="What is the capital of France?"
)
print(demo_response.text)

## 2. Load the CSV dataset
In Colab, upload only `finance_data.csv` from the repository's `data` folder when prompted. The notebook stops with a clear error if a different file is selected, then reads the CSV into a pandas DataFrame. A DataFrame is a table that Python can analyze.

In [ ]:
from google.colab import files

uploaded = files.upload()  # Select only finance_data.csv.
file_path = "finance_data.csv"

# Stop clearly if the expected dataset was not uploaded.
if file_path not in uploaded:
    raise FileNotFoundError("Upload finance_data.csv from the repository's data folder.")

df = pd.read_csv(file_path)
print("Dataset preview:")
display(df.head())
print("\nDataset dimensions:", df.shape)
print("\nColumn information:")
df.info()

## 3. Ask Gemini to analyze the data
This prompt gives Gemini the entire CSV and asks for trends, metrics, and observations. The output is natural-language analysis rather than a prediction table. The full dataset, including its target column, is included for analysis only.

In [ ]:
csv_string = df.to_csv(index=False)

analysis_prompt = f'''
Analyze the following dataset in CSV format.
Explain the main trends, important metrics, relationships between
columns, and any unusual observations. Use specific examples from
the data. Do not claim that correlation proves causation.
State that the dataset is synthetic.

DATA:
{csv_string}
'''

analysis_response = client.models.generate_content(
    model=MODEL_NAME, contents=analysis_prompt
)
print(analysis_response.text)

The next prompt tells Gemini which column is the target and uses both `company_id` and `date` to identify each row. Company IDs repeat across dates, so both fields are required for an unambiguous match. The prompt requests machine-readable CSV output so it can be loaded into pandas.

In [ ]:
# Exclude the actual target from model input to prevent target leakage.
prediction_csv = df.drop(columns=["stock_price"]).to_csv(index=False)

prediction_prompt = f'''
You are analyzing a synthetic practice dataset.
Predict the value of the "stock_price" column for every row using
the other available columns. Do not use the actual "stock_price"
value as an input when making the prediction.
Keep each row matched to its original "company_id" AND "date" because each company appears on multiple dates.
Return exactly three columns named "company_id", "date", and "predicted_stock_price".
Return one prediction for every input row.
Output ONLY valid CSV data: no explanation, no Markdown fences.

DATA:
{prediction_csv}
'''

prediction_response = client.models.generate_content(
    model=MODEL_NAME, contents=prediction_prompt
)
raw_prediction_text = prediction_response.text
print(raw_prediction_text)

## 5. Convert Gemini's CSV response into a DataFrame
This code removes optional Markdown fences, skips any leading explanation, and parses the CSV header and rows with pandas. It normalizes the two expected columns, converts the prediction to a number, and checks for duplicate, missing, or extra identifiers. A validated one-to-one merge then attaches predictions to the original rows. If Gemini omits or duplicates a row, the notebook stops with a clear error instead of silently reporting incomplete results.

In [ ]:
def response_to_dataframe(response_text):
    # Remove Markdown code fences if Gemini includes them.
    cleaned = response_text.strip()
    cleaned = re.sub(r"^```(?:csv)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    # Keep the response starting at the expected CSV header.
    lines = cleaned.splitlines()
    header_index = None
    for i, line in enumerate(lines):
        if "company_id" in line and "predicted_" in line:
            header_index = i
            break
    if header_index is None:
        raise ValueError("Could not find the expected CSV header in Gemini's response.")

    csv_text = "\n".join(lines[header_index:])
    return pd.read_csv(io.StringIO(csv_text))

predictions = response_to_dataframe(raw_prediction_text)
predictions.columns = predictions.columns.str.strip()
expected_prediction_column = "predicted_stock_price"
key_columns = ["company_id", "date"]

# Match the identifier, date, and prediction columns even if capitalization varies.
column_lookup = {c.lower(): c for c in predictions.columns}
if any(c not in column_lookup for c in key_columns + [expected_prediction_column]):
    raise ValueError("Expected columns: company_id, date, and predicted_stock_price")
source_columns = [column_lookup[c] for c in key_columns]
pred_match = column_lookup[expected_prediction_column]
predictions = predictions[source_columns + [pred_match]].rename(
    columns={**dict(zip(source_columns, key_columns)), pred_match: expected_prediction_column}
)
predictions["company_id"] = predictions["company_id"].astype(str).str.strip()
predictions["date"] = predictions["date"].astype(str).str.strip()
predictions[expected_prediction_column] = pd.to_numeric(
    predictions[expected_prediction_column], errors="coerce"
)
predictions = predictions.dropna(subset=[expected_prediction_column])

# A company appears on several dates, so the two-column key must be unique.
if predictions.duplicated(subset=key_columns).any():
    raise ValueError("Gemini returned duplicate company_id/date pairs.")
source_keys = set(map(tuple, df[key_columns].astype(str).to_numpy()))
prediction_keys = set(map(tuple, predictions[key_columns].to_numpy()))
missing_keys = source_keys - prediction_keys
extra_keys = prediction_keys - source_keys
if missing_keys or extra_keys or len(predictions) != len(df):
    raise ValueError(f"Prediction rows do not match the source data. Missing: {missing_keys}; extra: {extra_keys}")

# Join by both company and date to attach each estimate to the correct observation.
results = df.merge(predictions, on=key_columns, how="left", validate="one_to_one")
if results[expected_prediction_column].isna().any():
    raise ValueError("Some source rows do not have a valid prediction.")
display(results.head())
print("Rows with predictions:", len(results))

## 6. Compare actual values with AI predictions
The table compares the original target column with Gemini's predicted column. The mean absolute error (MAE) is the average absolute difference between actual and predicted values. Lower MAE means closer predictions on this dataset.

In [ ]:
results["absolute_error"] = (
    results["stock_price"] - results["predicted_stock_price"]
).abs()

print("Mean absolute error:", round(results["absolute_error"].mean(), 2))
display(results[["company_id", "stock_price", "predicted_stock_price", "absolute_error"]])

## 7. Plot actual versus predicted values
A separate chart panel is drawn for each company, with date on the horizontal axis and actual and predicted prices on the vertical axis. This preserves each company's monthly sequence and makes overestimates or underestimates easy to see.

In [ ]:
company_ids = sorted(results["company_id"].astype(str).unique())
fig, axes = plt.subplots(len(company_ids), 1, figsize=(12, 4 * len(company_ids)), squeeze=False)

# Draw a separate date-based comparison for each company.
for ax, company_id in zip(axes[:, 0], company_ids):
    company_data = results[results["company_id"].astype(str) == company_id].sort_values("date")
    ax.plot(company_data["date"], company_data["stock_price"],
            marker="o", label="Actual stock_price")
    ax.plot(company_data["date"], company_data["predicted_stock_price"],
            marker="x", label="AI predicted stock_price")
    ax.set_title(company_id)
    ax.set_xlabel("date")
    ax.set_ylabel("stock_price")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()

fig.suptitle("Actual vs. AI-Predicted stock_price by Company", y=1.01)
fig.tight_layout()
plt.show()

## 8. Save the results
This final cell saves the original data plus the AI predictions and errors as a new CSV file. In Colab, it also downloads the file to your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()  # Select only finance_data.csv.
file_path = "finance_data.csv"

# Stop clearly if the expected dataset was not uploaded.
if file_path not in uploaded:
    raise FileNotFoundError("Upload finance_data.csv from the repository's data folder.")

df = pd.read_csv(file_path)
print("Dataset preview:")
display(df.head())
print("\nDataset dimensions:", df.shape)
print("\nColumn information:")
df.info()

## Prompts used
1. **Data analysis prompt:** asks Gemini to explain trends, metrics, relationships, and unusual observations in the CSV.
2. **Prediction prompt:** asks Gemini to predict the target column for every row, preserve the row identifier, and return only CSV with the identifier and predicted value.

## Important limitation
The model's predictions can vary between runs and may be inaccurate. This notebook evaluates predictions against the same rows supplied to the model; it is a demonstration of prompt-based analysis, not a rigorous test of predictive performance on unseen data. A stronger evaluation would reserve a separate test set.